# Phase 24 — frozen cross-market robustness

This notebook tests whether the structural representation and GBP-selected model specification survive fixed market, period, sampling-frequency, and threshold-ladder changes. These symbols and histories have already been explored, so the results are robustness evidence—not untouched external confirmation.

In [1]:
import gc
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from dc_multiscale_research import (
    FeatureTier,
    ModelFamily,
    ObservationClock,
    add_future_diagnostics,
    build_multiscale_state,
    candidate_summary,
    eta_squared,
    filtering_prefix_invariant,
    fit_candidate,
    load_close,
    plot_price_states,
    prepare_data,
    ratio_ladder,
)

DATA_ROOT = Path(r"D:\data_dump\market_data\raw\FXPIG-Server\5")
MARKETS = {
    "GBPUSD.r": "fx",
    "EURUSD.r": "fx",
    "USDJPY.r": "fx",
    "US500.r": "equity_index",
    "XAUUSD.r": "metal",
}
PILOT_SYMBOL = "GBPUSD.r"
SAMPLING_MINUTES = (5, 15, 30)
LADDER_STARTS = (0.0015, 0.0020, 0.0025)
LADDER_RATIO = 1.5
THRESHOLD_COUNT = 5
PRIMARY_THRESHOLDS = ratio_ladder(0.0020, LADDER_RATIO, THRESHOLD_COUNT)
MINIMUM_OUTER_EVENTS = 75
CLOCK = ObservationClock.UNION_EVENT
FEATURE_TIER = FeatureTier.DYNAMIC
STATE_COUNTS = (3, 4, 5)
HMM_FAMILIES = (ModelFamily.NORMAL_HMM, ModelFamily.STUDENTT_HMM)
RANDOM_STATE = 17
MAX_ITER = 50
N_RESTARTS = 3
MINIMUM_VALIDATION_OCCUPANCY = 0.02
PLOT_RECENT_PRIMARY_CASES = True
LAST_N = 4_000

## Frozen robustness matrix

Structural sensitivity uses the full Cartesian product of five symbols, three close-sampling intervals, and three constant-ratio ladders. No case is dropped because its result looks inconvenient. Insufficient outer-event cases are reported rather than repaired with a symbol-specific threshold.

All symbols are restricted to their common available calendar interval. Fifteen- and thirty-minute closes are labelled at the right edge of their causal aggregation bucket.

In [2]:
native_prices = {symbol: load_close(DATA_ROOT, symbol) for symbol in MARKETS}
common_start = max(series.index[0] for series in native_prices.values())
common_end = min(series.index[-1] for series in native_prices.values())
print(f"Common interval: {common_start} to {common_end}")

market_contract = pd.DataFrame.from_records([
    {
        "symbol": symbol,
        "market_class": market_class,
        "native_bars": len(native_prices[symbol].loc[common_start:common_end]),
    }
    for symbol, market_class in MARKETS.items()
]).set_index("symbol")
display(market_contract)

Common interval: 2023-05-22 14:00:00+02:00 to 2026-11-09 13:30:00+02:00


,market_class,native_bars
symbol,,
GBPUSD.r,fx,258573
EURUSD.r,fx,258848
USDJPY.r,fx,258667
US500.r,equity_index,245467
XAUUSD.r,metal,244950


## Structural robustness across 45 predeclared cases

In [3]:
structural_records = []
for symbol, market_class in MARKETS.items():
    for sampling_minutes in SAMPLING_MINUTES:
        sampled = load_close(DATA_ROOT, symbol, sampling_minutes).loc[common_start:common_end]
        for ladder_start in LADDER_STARTS:
            thresholds = ratio_ladder(ladder_start, LADDER_RATIO, THRESHOLD_COUNT)
            _, global_state, _ = build_multiscale_state(
                sampled, thresholds, include_scale_state=False
            )
            thirds = pd.cut(
                np.arange(len(global_state)) / max(len(global_state) - 1, 1),
                bins=[0.0, 1.0 / 3.0, 2.0 / 3.0, 1.0],
                labels=["early", "middle", "late"],
                include_lowest=True,
            )
            by_third = global_state.assign(sample_third=thirds).groupby("sample_third", observed=False).agg(
                broad_share=("broad_agreement", "mean"),
                fragmented_share=("coherent_frontier", lambda values: 1.0 - values.mean()),
            )
            outer_events = int(global_state[f"dcc_flag_{THRESHOLD_COUNT - 1:02d}"].sum())
            structural_records.append({
                "symbol": symbol,
                "market_class": market_class,
                "sampling_minutes": sampling_minutes,
                "ladder_start": ladder_start,
                "outer_threshold": thresholds[-1],
                "initialized_bars": len(global_state),
                "observed_configurations": global_state["configuration_code"].nunique(),
                "broad_share": global_state["broad_agreement"].mean(),
                "fragmented_share": (~global_state["coherent_frontier"]).mean(),
                "union_event_share": global_state["union_event"].mean(),
                "outer_events": outer_events,
                "adequate_outer_events": outer_events >= MINIMUM_OUTER_EVENTS,
                "broad_share_period_range": by_third["broad_share"].max() - by_third["broad_share"].min(),
                "fragmented_share_period_range": (
                    by_third["fragmented_share"].max() - by_third["fragmented_share"].min()
                ),
            })
            del global_state
            gc.collect()

structural_robustness = pd.DataFrame.from_records(structural_records)
display(structural_robustness)
print("Cases:", len(structural_robustness))
print("Adequate outer-event cases:", structural_robustness["adequate_outer_events"].sum())

,symbol,market_class,sampling_minutes,ladder_start,outer_threshold,initialized_bars,observed_configurations,broad_share,fragmented_share,union_event_share,outer_events,adequate_outer_events,broad_share_period_range,fragmented_share_period_range
0,GBPUSD.r,fx,5,0.0015,0.007594,258007,32,0.263384,0.291810,0.031724,254,True,0.043077,0.059606
1,GBPUSD.r,fx,5,0.0020,0.010125,257813,32,0.263885,0.293709,0.019634,146,True,0.051863,0.044904
2,GBPUSD.r,fx,5,0.0025,0.012656,257639,32,0.268546,0.271780,0.013391,101,True,0.039151,0.014183
3,GBPUSD.r,fx,15,0.0015,0.007594,86219,32,0.290249,0.268131,0.073858,230,True,0.043459,0.032463
4,GBPUSD.r,fx,15,0.0020,0.010125,86152,32,0.284671,0.276198,0.048252,132,True,0.041857,0.023375
5,GBPUSD.r,fx,15,0.0025,0.012656,85637,32,0.284281,0.264232,0.034179,96,True,0.060545,0.028402
6,GBPUSD.r,fx,30,0.0015,0.007594,43194,32,0.304278,0.260013,0.119230,208,True,0.049937,0.057925
7,GBPUSD.r,fx,30,0.0020,0.010125,43168,32,0.300361,0.259776,0.081565,126,True,0.049204,0.022952
8,GBPUSD.r,fx,30,0.0025,0.012656,42910,32,0.290608,0.254416,0.058751,94,True,0.061176,0.021236
9,EURUSD.r,fx,5,0.0015,0.007594,258137,32,0.248922,0.303327,0.028942,238,True,0.027137,0.023159


Cases: 45
Adequate outer-event cases: 45


## Robustness summaries without case deletion

The first table asks whether market classes occupy structurally different hierarchies under the same grid. The second shows threshold and sampling sensitivity. Adequacy failures remain visible.

In [4]:
primary_cases = structural_robustness.loc[
    structural_robustness["sampling_minutes"].eq(5)
    & structural_robustness["ladder_start"].eq(0.0020)
]
display(primary_cases.set_index("symbol").style.format({
    "ladder_start": "{:.3%}",
    "outer_threshold": "{:.3%}",
    "broad_share": "{:.2%}",
    "fragmented_share": "{:.2%}",
    "union_event_share": "{:.2%}",
    "broad_share_period_range": "{:.2%}",
    "fragmented_share_period_range": "{:.2%}",
}))

sensitivity = structural_robustness.groupby(["sampling_minutes", "ladder_start"]).agg(
    cases=("symbol", "size"),
    adequate_cases=("adequate_outer_events", "sum"),
    median_outer_events=("outer_events", "median"),
    median_broad_share=("broad_share", "median"),
    median_fragmented_share=("fragmented_share", "median"),
)
display(sensitivity.style.format({
    "median_outer_events": "{:.1f}",
    "median_broad_share": "{:.2%}",
    "median_fragmented_share": "{:.2%}",
}))

,market_class,sampling_minutes,ladder_start,outer_threshold,initialized_bars,observed_configurations,broad_share,fragmented_share,union_event_share,outer_events,adequate_outer_events,broad_share_period_range,fragmented_share_period_range
symbol,,,,,,,,,,,,,
GBPUSD.r,fx,5,0.200%,1.012%,257813,32,26.39%,29.37%,1.96%,146,True,5.19%,4.49%
EURUSD.r,fx,5,0.200%,1.012%,258030,32,24.63%,31.00%,1.79%,144,True,2.69%,1.45%
USDJPY.r,fx,5,0.200%,1.012%,257979,32,32.65%,27.79%,2.75%,252,True,9.17%,2.33%
US500.r,equity_index,5,0.200%,1.012%,245113,32,33.57%,26.13%,5.26%,537,True,1.52%,0.63%
XAUUSD.r,metal,5,0.200%,1.012%,244719,32,28.37%,28.44%,8.55%,822,True,3.90%,3.03%


## Freeze the model family/state count on GBP validation

The specification is selected once on the pilot. External symbols do not choose their own family or number of states. Each external model may estimate emissions and transitions on its own training partition using the same algorithm and hyperparameters; this is frozen-procedure refitting, not direct parameter transfer.

In [5]:
pilot_prices = native_prices[PILOT_SYMBOL].loc[common_start:common_end]
_, pilot_global, _ = build_multiscale_state(
    pilot_prices, PRIMARY_THRESHOLDS, include_scale_state=False
)
pilot_prepared = prepare_data(pilot_global, CLOCK, FEATURE_TIER)
pilot_fits = []
pilot_selection_records = []
for family in HMM_FAMILIES:
    for n_states in STATE_COUNTS:
        fit = fit_candidate(
            pilot_prepared, family, n_states, RANDOM_STATE, MAX_ITER, N_RESTARTS
        )
        pilot_fits.append(fit)
        record = candidate_summary(fit, pilot_prepared)
        prefix_pass = filtering_prefix_invariant(fit, pilot_prepared)
        record["prefix_pass"] = prefix_pass
        record["eligible"] = (
            fit.error is None
            and prefix_pass is True
            and record.get("minimum_state_occupancy", 0.0) >= MINIMUM_VALIDATION_OCCUPANCY
        )
        pilot_selection_records.append(record)
pilot_selection = pd.DataFrame.from_records(pilot_selection_records)
pilot_eligible = pilot_selection.loc[pilot_selection["eligible"]]
if pilot_eligible.empty:
    raise RuntimeError("No pilot HMM passed the frozen eligibility rules.")
pilot_choice = pilot_eligible.sort_values("validation_log_likelihood_per_row", ascending=False).iloc[0]
SELECTED_FAMILY = ModelFamily(pilot_choice["family"])
SELECTED_N_STATES = int(pilot_choice["n_states"])
display(pilot_selection.set_index(["family", "n_states"]))
print(f"Frozen external procedure: {SELECTED_FAMILY.value}, K={SELECTED_N_STATES}")

clock     tier error  \
family       n_states                                    
normal_hmm   3         union_event_time  dynamic  None   
             4         union_event_time  dynamic  None   
             5         union_event_time  dynamic  None   
studentt_hmm 3         union_event_time  dynamic  None   
             4         union_event_time  dynamic  None   
             5         union_event_time  dynamic  None   

                       validation_log_likelihood_per_row  \
family       n_states                                      
normal_hmm   3                                -25.503625   
             4                                -23.770640   
             5                                -15.796295   
studentt_hmm 3                                -29.261737   
             4                                -18.936978   
             5                                -10.867945   

                       minimum_state_occupancy  switch_rate  median_dwell  \
family       n_states                                                       
normal_hmm   3                        0.308797     0.214029           4.0   
             4                        0.109515     0.329137           2.0   
             5                        0.109515     0.750000           1.0   
studentt_hmm 3                        0.308797     0.214029           4.0   
             4                        0.134650     0.570144           1.0   
             5                        0.129264     0.591727           1.0   

                       mean_max_probability  baseline_adjusted_rand  \
family       n_states                                                 
normal_hmm   3                     1.000000                0.466753   
             4                     0.997412                0.349851   
             5                     0.997405                0.784826   
studentt_hmm 3                     0.999987                0.466753   
             4                     1.000000                0.296693   
             5                     1.000000                0.482655   

                       baseline_nmi  baseline_mapped_agreement  prefix_pass  \
family       n_states                                                         
normal_hmm   3             0.568454                   0.651706         True   
             4             0.520404                   0.542190         True   
             5             0.879244                   0.791741         True   
studentt_hmm 3             0.568454                   0.651706         True   
             4             0.438379                   0.500898         True   
             5             0.603379                   0.630162         True   

                       eligible  
family       n_states            
normal_hmm   3             True  
             4             True  
             5             True  
studentt_hmm 3             True  
             4             True  
             5             True

Frozen external procedure: studentt_hmm, K=5


## Apply the frozen procedure to every primary market

The primary comparison holds sampling at five minutes and the threshold grid at 0.20% × 1.5. Model parameters are refitted on each market's training partition, but selection and evaluation rules remain fixed.

In [6]:
model_records = []
primary_paths = {}
for symbol, market_class in MARKETS.items():
    symbol_prices = native_prices[symbol].loc[common_start:common_end]
    _, symbol_global, _ = build_multiscale_state(
        symbol_prices, PRIMARY_THRESHOLDS, include_scale_state=False
    )
    symbol_prepared = prepare_data(
        symbol_global, CLOCK, FEATURE_TIER, split=pilot_prepared.split
    )
    fit = fit_candidate(
        symbol_prepared,
        SELECTED_FAMILY,
        SELECTED_N_STATES,
        RANDOM_STATE,
        MAX_ITER,
        N_RESTARTS,
    )
    record = candidate_summary(fit, symbol_prepared)
    record.update({"symbol": symbol, "market_class": market_class})
    if fit.error is None:
        test_states = fit.states["test"]
        test_occupancy = np.bincount(test_states, minlength=SELECTED_N_STATES) / len(test_states)
        diagnostics = add_future_diagnostics(symbol_global, (24,))
        test_frame = symbol_prepared.frames["test"].copy()
        test_frame["model_state"] = test_states
        test_frame = test_frame.join(diagnostics[["future_realized_vol_24", "future_dcc_count_24"]])
        record.update({
            "test_log_likelihood_per_row": fit.log_likelihoods["test"] / len(test_states),
            "test_minimum_occupancy": test_occupancy.min(),
            "test_switch_rate": np.mean(test_states[1:] != test_states[:-1]),
            "test_mean_confidence": fit.probabilities["test"].max(axis=1).mean(),
            "test_volatility_eta_squared": eta_squared(
                test_frame["future_realized_vol_24"], test_frame["model_state"]
            ),
            "test_activity_eta_squared": eta_squared(
                test_frame["future_dcc_count_24"], test_frame["model_state"]
            ),
            "prefix_invariant": filtering_prefix_invariant(fit, symbol_prepared),
        })
        path_parts = []
        for split_name in ("train", "validation", "test"):
            part = symbol_prepared.frames[split_name].copy()
            part["model_state"] = fit.states[split_name]
            part["model_semantic_code"] = fit.mapped_regimes[split_name]
            path_parts.append(part)
        primary_paths[symbol] = pd.concat(path_parts).sort_index()
    model_records.append(record)
    del symbol_global
    gc.collect()

model_robustness = pd.DataFrame.from_records(model_records).set_index("symbol")
display(model_robustness)

,family,n_states,clock,tier,error,validation_log_likelihood_per_row,minimum_state_occupancy,switch_rate,median_dwell,mean_max_probability,...,baseline_nmi,baseline_mapped_agreement,market_class,test_log_likelihood_per_row,test_minimum_occupancy,test_switch_rate,test_mean_confidence,test_volatility_eta_squared,test_activity_eta_squared,prefix_invariant
symbol,,,,,,,,,,,,,,,,,,,,,
GBPUSD.r,studentt_hmm,5,union_event_time,dynamic,None,-10.867945,0.129264,0.591727,1.0,1.000000,...,0.603379,0.630162,fx,-13.221069,0.107280,0.558541,1.000000,0.022654,0.005114,True
EURUSD.r,studentt_hmm,5,union_event_time,dynamic,None,-16.052404,0.018762,0.691729,1.0,0.998811,...,0.671390,0.789869,fx,-18.666326,0.015625,0.662192,0.997965,0.052084,0.026460,True
USDJPY.r,studentt_hmm,5,union_event_time,dynamic,None,-18.640409,0.014139,0.639640,1.0,0.997906,...,0.894606,0.877892,fx,-22.255168,0.064846,0.658120,0.997132,0.224039,0.151699,True
US500.r,studentt_hmm,5,union_event_time,dynamic,None,-16.015649,0.000716,0.650896,1.0,0.999445,...,0.997561,0.999284,equity_index,-15.933223,0.006565,0.650246,0.999227,0.031623,0.040360,True
XAUUSD.r,studentt_hmm,5,union_event_time,dynamic,None,-27.966403,0.122802,0.499038,1.0,0.996452,...,0.447661,0.707692,metal,-27.336808,0.102477,0.431629,0.994199,0.128313,0.125292,True


## Recent primary-market casebooks

Visual inspection asks whether inferred states correspond to intelligible price/hierarchy episodes in every market. These are regime markers, not trades.

In [7]:
if PLOT_RECENT_PRIMARY_CASES:
    for symbol, path in primary_paths.items():
        plot_price_states(
            path,
            "model_state",
            f"{symbol}: frozen-procedure causal states",
            last_n=LAST_N,
        ).show()
else:
    print("Recent cross-market casebook plots disabled by configuration.")

# Phase 24 completion gate

1. All 45 structural cases are retained, including inadequate cases.
2. Threshold ladders and frequencies are fixed across symbols rather than optimized per market.
3. Common calendar coverage is used.
4. Period instability is reported through early/middle/late occupancy ranges.
5. The HMM family and state count are selected on GBP validation only.
6. External markets refit the frozen procedure but do not reselect it.
7. Filtering causality, occupancy, confidence, persistence, and diagnostic usefulness are reported for every market.
8. Price-path casebooks must remain economically intelligible.
9. A result requiring different rules or stories for every symbol is non-transfer, not robustness.
10. These previously inspected markets cannot be represented as untouched external confirmation.